In [7]:
import sys
import os


if not os.path.exists("config.py"):
    os.chdir("backend") if os.path.exists("backend") else os.chdir("..")

sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())
print("config.py exists:", os.path.exists("config.py"))

Working directory: c:\Users\Dell\Documents\repo\llms\document-assistant\backend
config.py exists: True


In [9]:
import io
import re
import pdfplumber
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from googleapiclient.http import MediaIoBaseDownload
from config import settings

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
creds = Credentials.from_authorized_user_file(settings.google_token_file, SCOPES)
service = build("drive", "v3", credentials=creds)

PDF_FILE_ID = "1c-18lHMiW-wxFnUlhd6y21RNlP6ex_Ek"

def download_file(file_id: str) -> bytes:
    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return buffer.getvalue()

pdf_bytes = download_file(PDF_FILE_ID)

with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
    full_text = "\n".join(
        page.extract_text() for page in pdf.pages if page.extract_text()
    )

print(f"Text ready: {len(full_text):,} chars")

Text ready: 380,054 chars


In [10]:
# Print lines that look like clause headers
lines = full_text.split("\n")
header_lines = [l.strip() for l in lines if re.match(r'^(Clause|Sub-Clause|Article|Section|\d+\.\d+)', l.strip())]

print(f"Potential clause headers found: {len(header_lines)}")
print("\nFirst 30:")
for h in header_lines[:30]:
    print(" ", h)

Potential clause headers found: 1151

First 30:
  1.1 Definitions and Interpretation .............................................................................................. 1
  1.2 Contract Documents ........................................................................................................... 1
  1.3 Preliminary Matters ............................................................................................................. 1
  1.4 President as Authority’s Representative .............................................................................. 2
  1.5 Contractor’s Representative ................................................................................................ 2
  1.6 Communications ................................................................................................................. 2
  1.7 Contractor’s Responsibilities and Liabilities ......................................................................... 4
  1.8 Intellectual P

In [11]:
def clean_text(text: str) -> str:
    # Remove table of contents dot leaders e.g. "....... 12"
    text = re.sub(r'\.{4,}\s*\d+', '', text)
    # Collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    # Collapse more than 2 newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def chunk_by_clauses(text: str, max_tokens: int = 1500) -> list[dict]:
    text = clean_text(text)
    
    # Match clause headers like "1.1", "2.3", "12.4" at start of line
    pattern = re.compile(r'(?m)^(\d+\.\d+(?:\.\d+)?)\s+(.+)')
    
    matches = list(pattern.finditer(text))
    chunks = []

    for i, match in enumerate(matches):
        clause_ref = match.group(1)
        clause_title = match.group(2).strip()
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()

        # Skip TOC lines (very short content)
        if len(content) < 50:
            continue

        # Sub-chunk if too long
        words = content.split()
        if len(words) > max_tokens:
            step = max_tokens
            overlap = 100
            for j in range(0, len(words), step - overlap):
                sub_content = " ".join(words[j:j + step])
                chunks.append({
                    "clause_ref": f"{clause_ref} (part {j // (step - overlap) + 1})",
                    "clause_title": clause_title,
                    "content": sub_content,
                    "char_start": start,
                    "char_end": end,
                })
        else:
            chunks.append({
                "clause_ref": clause_ref,
                "clause_title": clause_title,
                "content": content,
                "char_start": start,
                "char_end": end,
            })

    return chunks


chunks = chunk_by_clauses(full_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 809


In [12]:
for chunk in chunks[5:10]:
    print(f"Clause: {chunk['clause_ref']} — {chunk['clause_title']}")
    print(f"Length: {len(chunk['content'].split())} words")
    print(chunk['content'][:300])
    print("---")

Clause: 2.10 — Authority’s Assistance
Length: 8 words
2.10 Authority’s Assistance 
3. CONTRACTOR’S GENERAL OBLIGATIONS 14
---
Clause: 3.2 — Contractor’s Execution
Length: 18 words
3.2 Contractor’s Execution 
Contents i
STATE OF QATAR DESIGN AND BUILD
PUBLIC WORKS AUTHORITY GENERAL CONDITIONS OF CONTRACT
---
Clause: 3.10 — Transportation of Goods and Customs Clearance
Length: 7 words
3.10 Transportation of Goods and Customs Clearance
---
Clause: 3.22 — Preparation and Implementation of Project Control Documents
Length: 8 words
3.22 Preparation and Implementation of Project Control Documents
---
Clause: 3.23 — Compliance of Project Control Documents with Contract
Length: 14 words
3.23 Compliance of Project Control Documents with Contract 
4. DESIGN - CONTRACTOR’S DOCUMENTS 22
---


In [13]:
full_text[5000:6000]

'........................................... 17\n3.9 Setting Out ........................................................................................................................ 17\n3.10 Transportation of Goods and Customs Clearance ............................................................. 17\n3.11 Co-ordination with Third Parties ........................................................................................ 18\n3.12 Contractor’s Operations on Site ........................................................................................ 18\n3.13 Security of the Site and Control of Access ........................................................................ 19\n3.14 Avoidance of Interference ................................................................................................. 19\n3.15 Access Route .................................................................................................................... 19\n3.16 Training .....................

In [14]:
# Find where the real content starts by looking for a clause with substantial text after it
for i, chunk in enumerate(chunks):
    if len(chunk['content'].split()) > 50:
        print(f"First substantial chunk at index {i}")
        print(f"Clause: {chunk['clause_ref']} — {chunk['clause_title']}")
        print(chunk['content'][:500])
        break

First substantial chunk at index 37
Clause: 1.2.1 — The following documents constitute the Contract Documents and shall be taken as
1.2.1 The following documents constitute the Contract Documents and shall be taken as
mutually explanatory of one another. For the purposes of interpretation of the Contract if
there is a conflict, ambiguity or discrepancy between these documents, the order of
precedence, from highest (a) to lowest (k) shall be as follows:
(a) Memorandum of Contract; and
(b) Particular Conditions; and
(c) General Conditions of Contract; and
(d) Schedule A [Project Brief]; and
(e) Schedule B [Payment Schedules]; 


In [15]:
short_chunks = [c for c in chunks if len(c['content'].split()) < 50]
print(f"Short chunks (under 50 words): {len(short_chunks)}")

long_chunks = [c for c in chunks if len(c['content'].split()) >= 50]
print(f"Substantial chunks (50+ words): {len(long_chunks)}")

# Check the boundary
for c in chunks[34:40]:
    print(f"{len(c['content'].split())} words — {c['clause_ref']}")

Short chunks (under 50 words): 371
Substantial chunks (50+ words): 438
17 words — 20.10
35 words — 20.12
45 words — 1.1.1
116 words — 1.2.1
105 words — 1.2.2
191 words — 1.3.1


In [16]:
# Check how many chunks contain dot leaders
toc_chunks = [c for c in chunks if '......' in c['content']]
real_chunks = [c for c in chunks if '......' not in c['content']]

print(f"TOC chunks: {len(toc_chunks)}")
print(f"Real chunks: {len(real_chunks)}")

# Verify a few from each
print("\nSample TOC chunks:")
for c in toc_chunks[:3]:
    print(f"  {c['clause_ref']} — {len(c['content'].split())} words")

print("\nSample real chunks:")
for c in real_chunks[:3]:
    print(f"  {c['clause_ref']} — {len(c['content'].split())} words")
    print(f"  {c['content'][:100]}")
    print()

TOC chunks: 0
Real chunks: 809

Sample TOC chunks:

Sample real chunks:
  1.8 — 7 words
  1.8 Intellectual Property (Contractor’s Documents and Information)

  1.16 — 6 words
  1.16 Extended Sub-Contractor Warranties and Guarantees

  1.17 — 8 words
  1.17 Parent Company Guarantee 
2. AUTHORITY’S OBLIGATIONS 9



In [17]:
with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
    for i in range(7, 12):  # pages 8-12 (0-indexed)
        text = pdf.pages[i].extract_text()
        print(f"\n--- Page {i+1} ---")
        print(text[:300] if text else "empty")


--- Page 8 ---
STATE OF QATAR DESIGN AND BUILD
PUBLIC WORKS AUTHORITY GENERAL CONDITIONS OF CONTRACT
20.11 Waiver ............................................................................................................................... 88
20.12 Assignment .....................................................

--- Page 9 ---
STATE OF QATAR DESIGN AND BUILD
PUBLIC WORKS AUTHORITY GENERAL CONDITIONS OF CONTRACT
1. THE CONTRACT AND PRELIMINARY MATTERS
1.1 Definitions and Interpretation
1.1.1 The defined words and expressions set out in Clause 1 of Appendix 1 [Definitions and
Interpretation] and the provisions relating to t

--- Page 10 ---
STATE OF QATAR DESIGN AND BUILD
PUBLIC WORKS AUTHORITY GENERAL CONDITIONS OF CONTRACT
(A) the Commencement Date; and
(B) the date upon which the Contractor starts any activities in
connection with the design or execution of the Works; and
(b) the Engineer has given its non-objection to the Initial C

--- Page 11 ---
STATE OF QATAR DESIGN AND BUILD

In [18]:
with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
    full_text = "\n".join(
        page.extract_text() for page in pdf.pages[8:]  # skip TOC (pages 1-8)
        if page.extract_text()
    )

print(f"Text ready: {len(full_text):,} chars")
print("\n--- First 500 chars ---")
print(full_text[:500])

Text ready: 354,436 chars

--- First 500 chars ---
STATE OF QATAR DESIGN AND BUILD
PUBLIC WORKS AUTHORITY GENERAL CONDITIONS OF CONTRACT
1. THE CONTRACT AND PRELIMINARY MATTERS
1.1 Definitions and Interpretation
1.1.1 The defined words and expressions set out in Clause 1 of Appendix 1 [Definitions and
Interpretation] and the provisions relating to the construction and interpretation of the
Contract set out in Clause 2 of Appendix 1 [Definitions and Interpretation] shall apply to
the Contract.
1.2 Contract Documents
1.2.1 The following documents 


In [19]:
def clean_text(text: str) -> str:
    # Remove table of contents dot leaders e.g. "....... 12"
    text = re.sub(r'\.{4,}\s*\d+', '', text)
    # Collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    # Collapse more than 2 newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def chunk_by_clauses(text: str, max_tokens: int = 1500) -> list[dict]:
    text = clean_text(text)
    
    # Match clause headers like "1.1", "2.3", "12.4" at start of line
    pattern = re.compile(r'(?m)^(\d+\.\d+(?:\.\d+)?)\s+(.+)')
    
    matches = list(pattern.finditer(text))
    chunks = []

    for i, match in enumerate(matches):
        clause_ref = match.group(1)
        clause_title = match.group(2).strip()
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()

        # Skip TOC lines (very short content)
        if len(content) < 50:
            continue

        # Sub-chunk if too long
        words = content.split()
        if len(words) > max_tokens:
            step = max_tokens
            overlap = 100
            for j in range(0, len(words), step - overlap):
                sub_content = " ".join(words[j:j + step])
                chunks.append({
                    "clause_ref": f"{clause_ref} (part {j // (step - overlap) + 1})",
                    "clause_title": clause_title,
                    "content": sub_content,
                    "char_start": start,
                    "char_end": end,
                })
        else:
            chunks.append({
                "clause_ref": clause_ref,
                "clause_title": clause_title,
                "content": content,
                "char_start": start,
                "char_end": end,
            })

    return chunks


chunks = chunk_by_clauses(full_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 773


In [20]:
for chunk in chunks[5:10]:
    print(f"Clause: {chunk['clause_ref']} — {chunk['clause_title']}")
    print(f"Length: {len(chunk['content'].split())} words")
    print(chunk['content'][:300])
    print("---")

Clause: 1.3.3 — Notwithstanding the effectiveness of the Contract and subject to Sub-clause 1.12
Length: 93 words
1.3.3 Notwithstanding the effectiveness of the Contract and subject to Sub-clause 1.12
[Advance Payment Guarantee] the Authority shall not be obliged to authorise or procure
any advance payments to the Contractor under the Contract until the Contractor has
provided the Authority with:
(a) a power of
---
Clause: 1.4.1 — The President shall represent the Authority in all matters relating to the Contract. Except
Length: 94 words
1.4.1 The President shall represent the Authority in all matters relating to the Contract. Except
for the duties expressly assigned to the Engineer under the Contract, the President shall
have the duties, powers and authority to act on behalf of and to bind the Authority for all
the purposes of the 
---
Clause: 1.4.2 — The President may, by notice to the Contractor, delegate and/or revoke some or all of its
Length: 49 words
1.4.2 The President may, by 

In [21]:
print(f"Total chunks: {len(chunks)}")
print(f"Shortest chunk: {min(len(c['content'].split()) for c in chunks)} words")
print(f"Longest chunk: {max(len(c['content'].split()) for c in chunks)} words")
print(f"Average chunk: {sum(len(c['content'].split()) for c in chunks) // len(chunks)} words")

Total chunks: 773
Shortest chunk: 6 words
Longest chunk: 931 words
Average chunk: 71 words


In [22]:
print("Key in settings:", settings.openai_api_key[:8], "...")

Key in settings: sk-... ...
